# Putting Words in GPT-2's Mouth with Sparse Autoencoders!

## Created by: Daniel Liu

### Check out my [LinkedIn](https://www.linkedin.com/in/daniel-sean-liu/), [GitHub](https://github.com/danielliu5670), and [HuggingFace](https://huggingface.co/danielliu1)!

In this file, I train a sparse autoencoder (SAE) on the twentieth layer of GPT-2 Medium, with the goal of "untangling" its nodes into monosemantic, interpretable features.

I then boost a chosen feature having to do with maritime and air travel, and show its impact on GPT-2 Medium's typical output (and how much it now loves boats).

I also include a sandbox you can use to play around with boosting your own features!

This is a demonstration of what would be considered a "basic" SAE, which means that its performance is much worse than the large-scale SAEs trained by Anthropic or OpenAI.

That said, I've created this project to show what would go into designing a basic SAE, which can then be improved on by using more modern techniques (such as JumpRELU, ghost gradients / more complex resampling, etc).

**Notes:**

*1. If you want to make any changes to this file (which you very likely will have to), you need to **make a copy of this Notebook in your Drive** first. To do this, click the "Copy to Drive" button in the header bar, next to "Run all."*

*2. Sections with the `(*) -` symbol preceding it are the most interesting, and I highly recommend going through and reading each one.*

*3. All sections with the `(!) -` symbol preceding it are necessary to run to set up for the aforementioned "interesting" parts of the Notebook.*

*4. All other sections are long-running code cells that extract and process data, which can be helpful to read through, but are not encouraged to run.*

*5. Running this file requires a lot of RAM. It is recommended to do so via Google Colaboratory - the T4 GPU runtime should provide just enough RAM for you to run the necessary files (`(!) -` and `(*) -`).*

***6. Understanding the design of this code requires a beginner to intermediate knowledge of the purpose of sparse autoencoders. For a more in-depth explanation on both of these topics, please visit [this link](https://adamkarvonen.github.io/machine_learning/2024/06/11/sae-intuitions.html) for a general overview.***

***7. Similarly, an intermediate knowledge of machine learning and concepts such as backpropagation, activations, etc. and a beginner knowledge of matrix operations and calculus is required.***  

***8. I am by no means an expert - in fact, there may be some inaccuracies in this code. If you notice anything that is wrong and need to be fixed, please send me a message via email (danielliu.datascience@gmail.com). Thank you!***

<div align="center">
  <img src="https://raw.githubusercontent.com/danielliu5670/SAE_GPT2/main/REALRating.png" width="100">
</div>

AI was used to assist with some of the coding (specifically when working with PyTorch and double-checking for logic errors at the end), but all of the explanations, annotations, design, and flow were created by me.

# (!) - Initialization

This code cell installs PyTorch and Numpy, both of which are used extensively for the machine learning parts of this Notebook.

In [2]:
!pip install torch
import torch
import numpy as np
import requests, io

This code cell mounts Google Drive, which connects your Google account so files can be saved. **This cell isn't necessary to run unless you plan to test out the data and activation extraction code yourself.**

In [3]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


Next, we begin by loading the same tokenizer that was used to train GPT-2 and loading the model itself. We specify `output_hidden_states` as True because we'll be analyzing the activations from these hidden layers. Otherwise, we would only be able to use this model for inference.

*Note: This process is faster if you have an `HF_TOKEN` secret in your Notebook. If you don't know how to do this, HuggingFace has several articles on it. Having this token isn't necessary for this code to work, however.*

In [4]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2-medium")

model = GPT2LMHeadModel.from_pretrained("gpt2-medium", output_hidden_states=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

This then loads the dataset WikiText, which is a much more manageable corpus (for my purposes) than OpenWebText or The Pile. Provided more storage and more resources (or if you, the viewer, has more storage or resources), a bigger dataset would almost certainly work better.

As a preliminary data processing step, any text that is less than 200 characters is removed to ensure any headers or other artifacts are not kept in the data, as well as to reduce the dimensionality of the dataset.

In [5]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
texts = []
for t in dataset["text"]:
    if len(t.strip()) > 200:
        texts.append(t)

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

# Activations Setup

Now, this code can be run to display some entries from the WikiText dataset. As you can see, each entry include the raw text of a Wikipedia article, which can often include artifacts or empty cells (which is why the pre-processing step above was necessary).

In [ ]:
print(texts[:5])

[' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n', " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . Character desig

We can now additionally see that there are a total of 14,313 entries in the dataset after processing.

In [ ]:
print(len(texts))

14313


Finally, we can define a function that extracts the activations from an input model (`model`), for a particular layer (specified with `layer_idx`) by iterating through the many training samples in the corpus (`corpus`).

These samples are each tokenized using the tokenizer defined above. These tokenized outputs are then passed into the model (whatever it may be). Notice that `torch.no_grad()` is used, since we only care about activation, and can save some computational effort by preventing backpropagation.

Afterward, we retrieve the activations from the layer we want to process, and squeeze out the batch dimension (since it doesn't hold valuable information to us). We also use `detach()` because we'll be working with the activations, and don't want it to affect the gradient computations that occur automatically with pyTorch.

Finally, we convert these tokens into a numpy array for convenience.

The output of this function is an N x M matrix, where N corresponds to the number of tokens present in the corpus, and M is the dimension of the layer specified by `layer_idx`.

In [ ]:
def get_act(model, layer_idx, corpus):

  inputs = tokenizer(corpus, return_tensors = "pt")

  with torch.no_grad():
    outputs = model(**inputs)

  layer_act_raw = outputs.hidden_states[layer_idx]
  layer_act = layer_act_raw.squeeze(0).detach().numpy()

  return layer_act

# Retrieving Activations

First off, we split the data into five batches, since the instance of Colab I was (am) running has too little RAM to run all of these activation extraction processes at once.

In [ ]:
batch1 = range(int(len(texts) / 5))
batch2 = range(int(len(texts) / 5), int(len(texts) / 5 * 2))
batch3 = range(int(len(texts) / 5 * 2), int(len(texts) / 5 * 3))
batch4 = range(int(len(texts) / 5 * 3), int(len(texts) / 5 * 4))
batch5 = range(int(len(texts) / 5 * 4), int(len(texts)))

Next, we can begin the actual process of extracting the activations from GPT-2. If you would like to run this file yourself, be sure to replace `activations3` and `batch3` to the batch you are currently processing.

Additionally, you can specify which of the layer you would like to analyze. To do this, change the second parameter of the `get_act()` function.

This process will take a long time, so be prepared to let it run in the background.

*Note: Running this requires you to have your Google Drive mounted to the Notebook. If you do not, or do not want to, it would be best not to run this cell, and instead only run the `(!) - ` sections.*

In [ ]:
activations = []

for i in range(len(texts)):
  text_act = get_act(model, 20, texts[i])
  activations.append(text_act)
  if i >= 100 and i % 100 == 0:
    print(str(i) + "/" + str(len(texts)))

activations = np.concatenate(activations, axis=0)
np.save("/content/drive/MyDrive/SAEDemo/activations.npy", activations)

100/14313
200/14313
300/14313
400/14313
500/14313
600/14313
700/14313
800/14313
900/14313
1000/14313
1100/14313
1200/14313
1300/14313
1400/14313
1500/14313
1600/14313
1700/14313
1800/14313
1900/14313
2000/14313
2100/14313
2200/14313
2300/14313
2400/14313
2500/14313
2600/14313
2700/14313
2800/14313
2900/14313
3000/14313
3100/14313
3200/14313
3300/14313
3400/14313
3500/14313
3600/14313
3700/14313
3800/14313
3900/14313
4000/14313
4100/14313
4200/14313
4300/14313
4400/14313
4500/14313
4600/14313
4700/14313
4800/14313
4900/14313
5000/14313
5100/14313
5200/14313
5300/14313
5400/14313
5500/14313
5600/14313
5700/14313
5800/14313
5900/14313
6000/14313
6100/14313
6200/14313
6300/14313
6400/14313
6500/14313
6600/14313
6700/14313
6800/14313
6900/14313
7000/14313
7100/14313
7200/14313
7300/14313
7400/14313
7500/14313
7600/14313
7700/14313
7800/14313
7900/14313
8000/14313
8100/14313
8200/14313
8300/14313
8400/14313
8500/14313
8600/14313
8700/14313
8800/14313
8900/14313
9000/14313
9100/14313
9200/143

# Concatenating Activations

This section is mostly a temporary / one-time measure to combine the batches created in the previous section. First, we simply load the numpy arrays that were saved to Drive.

In [ ]:
activations1 = np.load('/content/drive/MyDrive/SAEDemo/activations1.npy')

Next, we can take a look at what the activations look like before concatenating them. This is generally a good practice to make sure activations were not corrupted or incorrectly extracted during or after the process of uploading to Google Drive.

In [ ]:
activations1[0][:10]

array([ 1.1276941e+00,  5.3428006e-01,  3.5126865e-02, -8.4426820e-01,
       -1.7643958e-01,  2.9245441e+00, -1.1054609e+00,  1.4483852e+00,
        1.1533320e+00, -7.8215778e+02], dtype=float32)

Since these activations look good, we can similarly extract the other batches from Google Drive, and use `np.concatenate()` to combine them.

In [ ]:
activations2 = np.load('/content/drive/MyDrive/SAEDemo/activations2.npy')
activations3 = np.load('/content/drive/MyDrive/SAEDemo/activations3.npy')
activations4 = np.load('/content/drive/MyDrive/SAEDemo/activations4.npy')
activations5 = np.load('/content/drive/MyDrive/SAEDemo/activations5.npy')

In [ ]:
activations = np.concatenate((activations1, activations2, activations3, activations4, activations5), axis=0)

After combining these arrays, we can finally take a look at what the shape of the array is. From this, we can see there are several things of note we can learn:

Firstly, we can see that there were a total of 2,264,880 tokens in the WikiText dataset after tokenization. Secondly, we can see that the twentieth layer of GPT-2 seems to have 1,024 hidden nodes, since each column of this matrix corresponds to the activations produced by a single hidden node.

In [ ]:
activations.shape

(2264880, 1024)

We can then save this combined activations vector to Google Drive for future use.

In [ ]:
np.save("activations.npy", activations)

# Loading Activations

Firstly, we re-load the activations from Google Drive.

In [ ]:
activations_np = np.load('/content/drive/MyDrive/SAEDemo/activations.npy')

Once again - as a safety measure - it is usually a good practice to check the files we extract to ensure they have not been corrupted, and more importantly, do not contain any numbers that do not make sense.

In [ ]:
activations_np[0][:10]

array([ 1.1276956e+00,  5.3428036e-01,  3.5129532e-02, -8.4426516e-01,
       -1.7643651e-01,  2.9245427e+00, -1.1054612e+00,  1.4483848e+00,
        1.1533278e+00, -7.8215790e+02], dtype=float32)

Next, we can convert this numpy array into a tensor. The reason this was done is because, in PyTorch, the auto-grad functionality makes defining backpropagation much easier than if we were to define it by hand with numpy (which also likely would have been more inaccurate and less efficient).

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
activations = torch.tensor(activations_np, dtype=torch.float32, device=device)
activations_mean = activations.mean()
activations_std = activations.std()
activations = (activations - activations_mean) / activations_std
del activations_np
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
torch.save(activations, "/content/drive/MyDrive/SAEDemo/activations.pt")
torch.save(activations_mean, "/content/drive/MyDrive/SAEDemo/activations_mean.pt")
torch.save(activations_std, "/content/drive/MyDrive/SAEDemo/activations_std.pt")

# Functions

We can now begin by defining the functions that will be necessary for training the sparse autoencoder.

The first of these functions is `init_weights()`, which initializes the encoder and decoder weights of the SAE according to the input dimension (`input_dim`) of the hidden layer of interest, the "scale factor" (by which the input dimension should be expanded) (`scale`), and whether or not CUDA should be used. It is generally encouraged to use CUDA.

When initializing the encoder and decoder weight matrices, we specifically do so with small positive numbers, since using all zeros (like it was done with the bias vectors) would produce gradients that are always equivalent, for every node.

In [ ]:
def init_weights(input_dim, scale, device=None):
  if device is None:
      device = "cuda" if torch.cuda.is_available() else "cpu"

  hidden_dim = input_dim * scale

  enc_weights = (torch.randn((input_dim, hidden_dim), device=device) * 0.01).requires_grad_(True)
  enc_bias = torch.zeros((hidden_dim,), device=device, requires_grad=True)

  dec_weights = (torch.randn((hidden_dim, input_dim), device=device) * 0.01).requires_grad_(True)
  dec_bias = torch.zeros((input_dim,), device=device, requires_grad=True)

  return enc_weights, enc_bias, dec_weights, dec_bias

We also define a function that computes a forward pass through the sparse autoencoder using the input (`input`), encoder and decoder weights (`enc_weights` and `dec_weights`), and encoder and decoder bias (`enc_bias` and `dec_bias`).

This forward pass is comprised of the following steps:
1. Multiplying the input by the encoder weight matrix, and then adding the encoder bias.
2. Applying the ReLU activation function to enforce non-negativity and sparsity in the hidden layer.
3. Multiplying the hidden layer by the decoder weight matrix, and then adding the decoder bias.

Returned is the output, as well as the hidden layer itself (`input_out`). This is because the hidden layer is needed during backpropagation.

In [ ]:
def forward_pass(input, enc_weights, dec_weights, enc_bias, dec_bias):

  input_net = torch.matmul(input, enc_weights) + enc_bias
  input_out = torch.relu(input_net)

  output = torch.matmul(input_out, dec_weights) + dec_bias

  return output, input_out

We can also define a function that determines the total loss for a particular iteration by calculating the MSE of the output values (from the `forward_pass()` function) compared to the input.

There is also an added sparsity component, which is the mean of the absolute value of the hidden layer activations, multiplied by the sparsity coefficient (`sparsity_coef`).

The sparsity is divided by the batch size, since we are interested in the loss for a single sample, and not the entire batch.

In [ ]:
def loss(input, output, hidden, sparsity_coef, batch_size):

  mse = torch.nn.functional.mse_loss(output, input)
  sparsity = (sparsity_coef * torch.abs(hidden).sum()) / batch_size
  return mse + sparsity

Our last function is designed to combat what's typically known as feature death, using "dead feature resampling."

Because the L1 sparsity penalty pushes feature activations toward zero, some features may become inactive ("dead") during training, which produces a final set of features that are all identical and, basically, useless.

It's difficult to predict when this will happen because the L1 loss, which is one of the metrics we're optimizing (which measures the total magnitude of the features, unlike the L0, which we see), doesn't distinguish between what features are used for what.

For example, think of it this way: if we have a very small set of features that are active, but are *very* active, then the L1 will treat it the same as having a huge set of features that are active, but not quite as active.

The L0 won't tell you necessarily whether you need resampling, since every token using the same 80 features and every token using a different set of 80 features will both produce an average L0 of 80.

The way resampling works is that, every so often (how often I'll specify in the next text cell), the inputs that have the highest loss are identified, and then the dead features are pointed toward them. They're initialized at one-fifth the strength of the alive features, because it can be bad if the dead features are initialized as too weak or too strong relative to the alive.

Essentially, each one is given a new chance at doing something useful.

The optimizer is also passed into this function because, otherwise, Adam would "push" the feature back into being dead otherwise (using the momentum from when the feature was previously dead).

*Note: As I'll discuss later, this doesn't at all fix the dead features problem, especially in this case when the input dataset is so small, but it definitely can help at scale, as well as if it's combined with other advanced methods. For the sake of this notebook, I won't include them.*

In [ ]:
def resample(enc_w, dec_w, enc_b, dec_b, fire_ct, batch, optimizer):
    dead = (fire_ct == 0)
    n_dead = dead.sum().item()
    if n_dead > 0:
        print("resampled:", n_dead)
    if n_dead == 0:
        return

    with torch.no_grad():
        out, _ = forward_pass(batch, enc_w, dec_w, enc_b, dec_b)

    per_loss = (batch - out).pow(2).sum(dim=-1)
    probs = per_loss / per_loss.sum()
    sampled = torch.multinomial(probs, num_samples=n_dead, replacement=True)

    dirs = batch[sampled]
    dirs = dirs / (dirs.norm(dim=-1, keepdim=True) + 1e-8)

    alive = ~dead
    avg_enc_norm = enc_w.data[:, alive].norm(dim=0).mean()
    avg_dec_norm = dec_w.data[alive, :].norm(dim=1).mean()

    enc_w.data[:, dead] = dirs.T * avg_enc_norm * 0.2
    enc_b.data[dead] = 0.0
    dec_w.data[dead, :] = dirs * avg_dec_norm * 0.2

    params = optimizer.param_groups[0]["params"]
    for param_idx, param in enumerate(params):
        if param not in optimizer.state:
            continue
        state = optimizer.state[param]
        if "exp_avg" not in state:
            continue
        if param_idx == 0:
            state["exp_avg"][:, dead] = 0.0
            state["exp_avg_sq"][:, dead] = 0.0
        elif param_idx == 1:
            state["exp_avg"][dead] = 0.0
            state["exp_avg_sq"][dead] = 0.0
        elif param_idx == 2:
            state["exp_avg"][dead, :] = 0.0
            state["exp_avg_sq"][dead, :] = 0.0

Finally, we're able to put all of these elements together into one function that performs the entire SAE training process, which uses PyTorch's built-in Adam optimizer.

At its core, this function initializes the optimizer with all four of our trainable tensors (encoder weights, encoder bias, decoder weights, decoder bias), and then iterates through each batch of samples. For each batch, it does a forward pass (using `forward_pass()`), computes the loss (using `loss()`), and then calls `optimizer.zero_grad()`, `total_loss.backward()`, and `optimizer.step()` to perform the computations and weight updates automatically.

After each optimizer step, the decoder weight rows are constrained to have a maximum norm of 1.0, which prevents the optimizer from cheating by moving the encoder and decoder weights in opposite directions.

We also keep track of the number of active features (for resampling purposes). Currently, every 1000 batches (so approximately once per iteration), resampling is performed, and then the number of resampled, or previously dead features, is printed out.

After each epoch, the function prints out the iteration number, MSE value, and the L0 sparsity. After the training is complete, the final encoder and decoder weights and encoder and decoder bias vectors are returned.

*Note: There are a few things that can be tuned here: 1.) the frequency of resampling, currently 1000, 2.) the batch size, 3.) the resampling batch size.*

In [ ]:
def train(activations, scale, sparsity_coef, learning_rate, iterations):

  input_dim = activations.shape[1]
  samples = activations.shape[0]
  test = activations[:1000]
  batch_size = 2048
  mse_lst = []
  fire_ct = torch.zeros(input_dim * scale, device=activations.device)

  enc_weights, enc_bias, dec_weights, dec_bias = init_weights(input_dim, scale)

  optimizer = torch.optim.Adam(
    [enc_weights, enc_bias, dec_weights, dec_bias],
    lr=learning_rate
  )

  for i in range(iterations):
    shuffled_act = activations[torch.randperm(activations.shape[0])]
    for j in range(0, samples, batch_size):
      if j + batch_size < samples:
        batch = shuffled_act[j:j + batch_size]

        output, hidden = forward_pass(batch, enc_weights, dec_weights, enc_bias, dec_bias)
        total_loss = loss(batch, output, hidden, sparsity_coef, batch_size)

        optimizer.zero_grad()
        total_loss.backward()
        fire_ct += (hidden > 0).float().sum(dim=0)
        optimizer.step()

        with torch.no_grad():
            norms = dec_weights.data.norm(dim=1, keepdim=True)
            dec_weights.data.div_(norms.clamp(min=1.0))

        if j % (batch_size * 1000) == 0 and j > 0 and i >= 2:
            resample_batch = activations[torch.randperm(activations.shape[0])[:4096]]
            resample(enc_weights, dec_weights, enc_bias, dec_bias, fire_ct, resample_batch, optimizer)
            fire_ct.zero_()

    with torch.no_grad():
        output_test, hidden_test = forward_pass(test, enc_weights, dec_weights, enc_bias, dec_bias)
        mse_test = torch.nn.functional.mse_loss(output_test, test)
        mse_lst.append(mse_test)
        l0_test = (hidden_test > 0.001).float().sum(dim=1).mean()
        print("iter:", i, "| mse:", mse_test.item(), "| l0:", l0_test.item())
    if len(mse_lst) >= 5 and all(abs(mse_lst[-j] - mse_lst[-j-1]) <= 0.0001 for j in range(1, 5)):
        break

  return enc_weights, dec_weights, enc_bias, dec_bias

# Training

Now, we can use our `train()` function to extract the weight matrices and bias vectors.

Note that the hyperparameters I've set (the scale, sparsity coefficient, learning rate, and number of iterations) are all things that can be modified at will. While I found these specific hyperparameters to be the best and highest-performing, they are by no means objectively the best.

In [ ]:
activations = torch.load("/content/drive/MyDrive/SAEDemo/activations.pt")

There are a few things to note with the outputs of the training function: *mostly if you're interested in retraining the SAE or want to know how I decided what is considered a "good" or high-quality SAE*.

Firstly, the MSE represents the reconstruction error of the inputs. Therefore, you want it to be as close to 0 as possible. Specifically, I've noticed that the best MSE value compared to the L0 (which I'll discuss in a moment) tends to be around 0.08, but if you can get a better score, that would be even better.

Secondly, the L0 should be somewhere between 50 to 100. This is a pretty baseline number that is used by many of the papers / research I looked at in preparation. The higher the L0, generally, the lower the MSE, but for the purposes of this Notebook, I would prefer the L0 to be lower.

The `resampled:` number should, ideally, be decreasing. This would mean that the dead features being resampled are *probably* not repeatedly dying. This `resampled:` number does NOT, however, tell you how many features are not "dead." The reasons for this are a bit complex, but essentially, the resampling method I use is a bit naive.

In [ ]:
enc_weights_fnl, dec_weights_fnl, enc_bias_fnl, dec_bias_fnl = train(activations, 4, 0.0007, 0.0003, 9)

iter: 0 | mse: 0.09353400021791458 | l0: 67.4010009765625
iter: 1 | mse: 0.08579295128583908 | l0: 65.09300231933594
iter: 2 | mse: 0.08207253366708755 | l0: 68.73900604248047
resampled: 2276
iter: 3 | mse: 0.1284075826406479 | l0: 32.96000289916992
resampled: 661
iter: 4 | mse: 0.08581172674894333 | l0: 53.9630012512207
resampled: 338
iter: 5 | mse: 0.07946053147315979 | l0: 60.96900177001953
resampled: 148
iter: 6 | mse: 0.07554304599761963 | l0: 67.82200622558594
resampled: 52
iter: 7 | mse: 0.07303234934806824 | l0: 74.27100372314453
resampled: 31
iter: 8 | mse: 0.07060394436120987 | l0: 79.61700439453125


After the training is complete, we can save all of our output tensors into `.pt` files in Google Drive.

In [ ]:
torch.save(enc_weights_fnl, "/content/drive/MyDrive/SAEDemo/enc_weights_fnl.pt")
torch.save(dec_weights_fnl, "/content/drive/MyDrive/SAEDemo/dec_weights_fnl.pt")
torch.save(enc_bias_fnl, "/content/drive/MyDrive/SAEDemo/enc_bias_fnl.pt")
torch.save(dec_bias_fnl, "/content/drive/MyDrive/SAEDemo/dec_bias_fnl.pt")

# (!) - Evaluation Setup

First, we load all of the tensors.

*Note: From here on out, I will be pulling the data files from my HuggingFace dataset repository, so anyone who needs to run these portions of the notebook will be able to.*

In [64]:
device = "cuda" if torch.cuda.is_available() else "cpu"

enc_weights_fnl  = torch.load(io.BytesIO(requests.get("https://huggingface.co/datasets/danielliu1/SAEDemo/resolve/main/enc_weights_fnl.pt").content), map_location=device)
dec_weights_fnl  = torch.load(io.BytesIO(requests.get("https://huggingface.co/datasets/danielliu1/SAEDemo/resolve/main/dec_weights_fnl.pt").content), map_location=device)
enc_bias_fnl     = torch.load(io.BytesIO(requests.get("https://huggingface.co/datasets/danielliu1/SAEDemo/resolve/main/enc_bias_fnl.pt").content), map_location=device)
dec_bias_fnl     = torch.load(io.BytesIO(requests.get("https://huggingface.co/datasets/danielliu1/SAEDemo/resolve/main/dec_bias_fnl.pt").content), map_location=device)
activations_mean = torch.load(io.BytesIO(requests.get("https://huggingface.co/datasets/danielliu1/SAEDemo/resolve/main/activations_mean.pt").content), map_location=device)
activations_std  = torch.load(io.BytesIO(requests.get("https://huggingface.co/datasets/danielliu1/SAEDemo/resolve/main/activations_std.pt").content), map_location=device)

Then, we can begin to use our newly-formed weight matrices and bias vectors to test out some input values and how their hidden layers end up looking.

 To do this, we can use the below function, which passes in some text through the original model (not the SAE, so GPT-2, in this case) to get activations from some particular layer.

Afterward, those activations are passed through the sparse autoencoder's encoder weights to get the hidden layer values. Since there are (probably) more than one token in the input text, the output will be an N x M matrix, where N represents the number of tokens in the input text, and M is the number of expanded hidden-layer activations from the sparse autoencoder.

Specifically, this function is very similar to the activation extraction function that was defined toward the beginning of this Notebook. One major difference, however, is that these activations must be normalized, since this was a component of training the SAE.

In [65]:
def get_ft_act(text_tk, model, layer_idx, enc_weights, enc_bias, act_mean, act_std):
  device = "cuda" if torch.cuda.is_available() else "cpu"

  with torch.no_grad():
    outputs = model(**text_tk)

  layer_act_raw = outputs.hidden_states[layer_idx]
  layer_act = layer_act_raw.squeeze(0).detach().float().to(device)

  layer_act = (layer_act - act_mean) / act_std

  layer_act_pass = torch.matmul(layer_act, enc_weights) + enc_bias
  layer_act_fnl = torch.relu(layer_act_pass)

  return layer_act_fnl

We can now test out this function by passing in some test text. In this case, we can see that the tokenizer, when dividing up the phrase "The capital of France is Paris.", produces seven tokens.

After the application of the `get_ft_act()` function, each of these tokens gets a column corresponding to a feature in the expanded hidden layer.

This particular number of 4,096 features makes sense because, as you may remember, GPT-2's twentieth layer had 1,024 total nodes. A scale of 4 (which was a parameter of the training function) would produce 4,096 expanded features.

In [66]:
device = "cuda" if torch.cuda.is_available() else "cpu"

test = "The capital of France is Paris."
test_tk = tokenizer(test, return_tensors="pt").to(device)
model = model.to(device)
features = get_ft_act(test_tk, model, 20, enc_weights_fnl, enc_bias_fnl, activations_mean, activations_std)
features.shape

torch.Size([7, 4096])

We also define a function, `top_tk()`, allows the user to specify a feature by its index and find out which of the tokens from the text (from get_ft_act(), not get_act()) caused it to most strongly activate.

The output is a list of length K (specified by top_k), where each element is a tuple of the token and its associated activation. The reason we extract these tuples will become more clear in the following section, where we provide an example of the first stage output.

In [67]:
def top_tk(feature_idx, activations, text_tk, top_k):
  ft_act = activations[:, feature_idx]
  values, indices = torch.sort(ft_act, descending=True)

  top_k_tups = []

  for i in range(top_k):
    value = values[i].item()
    index = indices[i]
    token = text_tk[index]
    top_k_tups.append((token, value))

  return top_k_tups

Finally, we define a function to extract only the token, not its activation. It is identical in logic to the `top_tk()` function otherwise.

In [68]:
def top_tk_only(feature_idx, activations, text_tk, top_k):
  ft_act = activations[:, feature_idx]
  values, indices = torch.sort(ft_act, descending=True)

  top_tk = []

  for i in range(top_k):
    index = indices[i]
    token = text_tk[index]
    top_tk.append(token)

  return top_tk

# WikiText Activations

Now, we can begin a demonstration of how our sparse autoencoder can extract (hopefully) meaningful features from the WikiText dataset (which, if you remember, we used to train the SAE in the first place). While this might introduce some element of overfitting, if the result is some set of informative features, that's good enough for now. Perhaps a future project could explore the generalizability of my SAE.

This section, specifically, focuses on extracting activations.

We begin by creating lists to store our WikiText tokens and activations, since both will be used later.

After this, we begin iterating through the first 500 entries of the original WikiText dataset (which, if you remember from the beginning of this Notebook, is being stored in the `texts` variable). The reason I've chosen 500 is because, simply put... storing and analyzing the entire dataset would have overwhelmed my computer.

For anyone wanting to run this code themselves, this number can certainly be adjusted.

During this iteration, we go through the following process:

1. First, we tokenize the inputs using the same tokenizer (GPT-2).
2. Next, we extract the feature activations from the model, using the twentieth layer (which can also be customized... as long as the SAE also was trained on the same layer).
3. We then squeeze out the IDs and add the token IDs to the token list. We also add the features.
4. We then convert the IDs back into readable tokens.
5. Finally, we print out some progress indicators.

In [ ]:
wikitext_tk = []
wikitext_act = []
count = 0

for text in texts[:500]:
    text_tk = tokenizer(text, return_tensors="pt").to(device)
    features = get_ft_act(text_tk, model, 20, enc_weights_fnl, enc_bias_fnl,
                          activations_mean, activations_std)
    token_ids = text_tk["input_ids"].squeeze(0)
    for tid in token_ids:
        wikitext_tk.append(tokenizer.decode([tid]))
    wikitext_act.append(features.cpu())
    count += 1
    if count >= 50 and count % 50 == 0:
        print(count, "/", 500)

wikitext_act = torch.cat(wikitext_act, dim=0)

50 / 500
100 / 500
150 / 500
200 / 500
250 / 500
300 / 500
350 / 500
400 / 500
450 / 500
500 / 500


Once this is done, we save our tensors to Drive to use for later.

In [ ]:
torch.save(wikitext_act, "/content/drive/MyDrive/SAEDemo/wikitext_act.pt")

In [ ]:
torch.save(wikitext_tk, "/content/drive/MyDrive/SAEDemo/wikitext_tk.pt")

# (*) - Evaluation Pt. I

Now, we can begin analyzing the activations we've painstakingly extracted from the WikiText dataset. To do this, we first need to load the raw activations and tokens from the tensor files we've created to store them.

In [24]:
device = "cuda" if torch.cuda.is_available() else "cpu"

wikitext_act = torch.load(io.BytesIO(requests.get("https://huggingface.co/datasets/danielliu1/SAEDemo/resolve/main/wikitext_act.pt").content), map_location=device)
wikitext_tk = torch.load(io.BytesIO(requests.get("https://huggingface.co/datasets/danielliu1/SAEDemo/resolve/main/wikitext_tk.pt").content), map_location=device)

We can also verify that the data is in the correct format by using the `.shape` attribute - in fact, we can see that there were a total of 82,764 tokens, and that the dimension of the expanded GPT-2 layer is 4,096 features. The length of the `wikitext_tk` array should exactly match the number of rows in `wikitext_act`, or else something has gone wrong.

In [25]:
wikitext_act.shape

torch.Size([82764, 4096])

In [26]:
len(wikitext_tk)

82764

Now, we can call the `top_tk_only()` function we defined in the previous section. Using the following set of parameters (which you should feel free to change and play around with), we are asking the question:

*For the `2398`th feature, while using the `wikitext_act` set of activations and `wikitext_tk` set of tokens, what are the top `10` tokens that caused this feature to most strongly activate?*

As we can see, I've chosen this feature in particular because it activates for a very specific concept almost exclusively - Christianity.

In [36]:
top_tk_tups = top_tk_only(2397, wikitext_act, wikitext_tk, 10)
print(top_tk_tups)

[' God', ' Christ', ' god', ' gods', ' Christ', ' Christ', ' God', ' Pope', ' god', ' gods']


At this point, I was interested to see (and maybe you are too) whether I could scroll through a bunch of features and try and theorize what they represented at a high level.

Thankfully, we can simply use a `for` loop to do this.

**I've created the `start_ft` and `end_ft` features for you to play around with this cell. You can use them to specify the start and end (respectively) of any subset of the features you would like to analyze. I've initialized them with what I think is the most interesting and salient subset.**

**For any programmers,** I've adjusted the `range()` indices so the start and end variables are 1-indexed. I figure this would be easier to understand. The feature numbers in the output are 1-indexed, as well.

*Note: You may notice that many of the feature are missing. This is because, despite my best efforts, the input WikiText dataset is simply too tiny for all of the features to **not** be dead. In fact, around 86% of the features are dead. Despite this, the alive features are really interesting and interpretable, so that's what I've chosen to show here. Some of the most advanced SAEs trained on datasets that are thousands of times larger still have at least some percentage of dead features, so I'd say that for a tiny input dataset, I'm doing pretty good!*

In [91]:
start_ft = 2360
end_ft = 2500

for i in range(start_ft - 1, end_ft):
    if wikitext_act[:, i].max().item() > 0.24:
        top_tk_tups = top_tk_only(i, wikitext_act, wikitext_tk, 10)
        print("feature", i + 1, ":", top_tk_tups)

feature 2363 : [' spent', ' during', ' spent', ' spent', ' spent', ' spent', ' during', ' During', ' spending', ' over']
feature 2364 : [' has', ' had', ' had', ' had', ' had', ' had', ' had', ' had', ' has', ' had']
feature 2367 : [' of', ' of', ' commit', ' for', ' of', ' acts', ' of', ' previous', ' of', ' were']
feature 2368 : [' located', ' located', ' located', ' located', ' position', ' stationed', ' placed', ' located', ' stationed', 'ed']
feature 2372 : [' 250', '000', ' 300', ' 400', ' 130', ' 200', ' 240', ' 800', '91', ' 150']
feature 2377 : [' tight', ' a', ' wear', ' fitting', ' long', 'earing', ' civilian', ' of', ' ,', ' wear']
feature 2381 : [' begins', ' began', ' began', ' begins', ' began', ' begin', ' began', ' began', ' began', ' begins']
feature 2388 : [' There', ' There', ' there', ' There', ' there', ' There', ' there', ' there', ' There', ' there']
feature 2398 : [' God', ' Christ', ' god', ' gods', ' Christ', ' Christ', ' God', ' Pope', ' god', ' gods']
featu

In [60]:
alive_ct = 0
for i in range(4096):
    if wikitext_act[:, i].max().item() > 0.24:
        alive_ct += 1

print("% of alive features:", alive_ct / 4096)

% of alive features: 0.140380859375


# (!) - Modifying Output (Functions)

In [69]:
device = "cuda" if torch.cuda.is_available() else "cpu"

wikitext_act = torch.load(io.BytesIO(requests.get("https://huggingface.co/datasets/danielliu1/SAEDemo/resolve/main/wikitext_act.pt").content), map_location=device)
wikitext_tk = torch.load(io.BytesIO(requests.get("https://huggingface.co/datasets/danielliu1/SAEDemo/resolve/main/wikitext_tk.pt").content), map_location=device)

Now, we can finally get to the meat and potatoes - that is, proving that boosting a particular token actually has a corresponding effect on the model's outputs.

For the sake of demonstration, in this section, I will use feature 369, which seems to correspond to *maritime travel* - boats, ports, the ocean, the sea, etc. Our hypothesis will be that boosting this feature will cause the model to output maritime-related phrases more frequently.

In [70]:
top_tk_tups = top_tk_only(368, wikitext_act, wikitext_tk, 10)
print(top_tk_tups)

[' ocean', ' sea', ' port', ' sea', ' port', ' shore', ' naval', ' naval', ' naval', ' port']


First, we need to define a function that will be attached to the forward hook (what this is I will explain in the next cell).

Essentially, what the function does is that, first, it takes its input (which would be the 20th layer's activations) and "passes it through" the first layer of the sparse autoencoder by multiplying it by the encoder weights, adding the encoder bias, and then apply ReLU.

Think of it as turning the activations into the expanded, interpretable features we just looked at.

Then, our feature of interest will be boosted by a customizable amount, and then it will "passed through" the rest of the SAE. Think of this as "reconstructing" the original input, but modified so that our feature of interest has a huge impact on what comes out.

In [71]:
def boost(module, layer_in, layer_out):
    hidden = layer_out[0]

    hidden_norm = (hidden - module.act_mean) / module.act_std

    feats = torch.relu(torch.matmul(hidden_norm, module.enc_w) + module.enc_b)
    feats[:, :, module.feat_idx] += module.boost_amt
    mod_hidden = torch.matmul(feats, module.dec_w) + module.dec_b

    mod_hidden = mod_hidden * module.act_std + module.act_mean

    return (mod_hidden,) + layer_out[1:]

Next, we define our parent function, which is what we will be calling to perform our feature boosting.

In this function, the first thing we do (after initializing all of the variables we'll need) is register a forward hook for our 20th layer.

A forward hook, at its core, is something we can do to the outputs of a particular layer. It can be used in many different ways - a forward hook can be used to observe these outputs and then document them, as well as (in this case) modify them.

Since the forward hook itself takes in a function as a paramter, we can use the `boost()` function we just defined and attach it.

We then generate the modified output using the `torch.no_grad()` function (since we don't care about the backpropagation process), remove the hook, and then return the output.

In [72]:
def boost_gen(prompt, model, tk, enc_w, enc_b, dec_w, dec_b, act_mean, act_std, feat_idx, boost_amt):
    layer = model.transformer.h[19]

    layer.enc_w = enc_w
    layer.enc_b = enc_b
    layer.dec_w = dec_w
    layer.dec_b = dec_b
    layer.act_mean = act_mean
    layer.act_std = act_std
    layer.feat_idx = feat_idx
    layer.boost_amt = boost_amt

    hdl = layer.register_forward_hook(boost)

    in_tk = tk(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **in_tk,
            max_new_tokens=1,
            pad_token_id=tk.eos_token_id,
            do_sample=True
        )

    hdl.remove()
    del layer.enc_w, layer.enc_b, layer.dec_w, layer.dec_b, layer.act_mean, layer.act_std, layer.feat_idx, layer.boost_amt

    return out_ids

# (*) - Modifying Output (Example)

Since GPT-2 is a completion model and not an assistant model (like most modern GPTs), our prompt has to be a "fill-in-the-rest" type. I'm most interested in seeing what boosting this feature does to the model's preference for travel. Who knows, maybe it really likes boats already!

To do this, I'll use the following prompt. We also initialize a dictionary to store the distribution of outputs, and tokenize our prompt to send it into the model.

In [ ]:
prompt1 = "The best way to travel is by using a"
base_dist = {}
in_tk = tokenizer(prompt1, return_tensors="pt").to(model.device)

Then, what I'll do is collect 100 of the model's outputs and convert them into readable text.

In [ ]:
for _ in range(100):
    with torch.no_grad():
        out_ids = model.generate(**in_tk, max_new_tokens=1, pad_token_id=tokenizer.eos_token_id, do_sample=True)

    new_id = out_ids[0][-1]
    txt = tokenizer.decode(new_id).strip()
    base_dist[txt] = base_dist.get(txt, 0) + 1

Finally, we can take a look at the distribution of outputs for this prompt.

It seems like the model, without boosting, prefers traveling by car and bicycle, with some others mixed in (taxi having appeared four times!). Of course, GPT-2 Medium isn't the most intelligent model, so there are some nonsense answers (budget?).

In [ ]:
sort_base = dict(sorted(base_dist.items(), key=lambda x: x[1], reverse=True))

for word, pct in sort_base.items():
    print(f"{word}: {pct / sum(sort_base.values()) * 100:.1f}%")

car: 18.0%
bicycle: 13.0%
bike: 10.0%
mobile: 5.0%
taxi: 4.0%
vehicle: 4.0%
motorcycle: 4.0%
plane: 4.0%
train: 3.0%
bus: 3.0%
smartphone: 3.0%
boat: 3.0%
wheelchair: 2.0%
map: 2.0%
carrier: 2.0%
single: 2.0%
shuttle: 1.0%
van: 1.0%
shared: 1.0%
tour: 1.0%
walk: 1.0%
transit: 1.0%
rail: 1.0%
good: 1.0%
coach: 1.0%
motor: 1.0%
travel: 1.0%
combination: 1.0%
budget: 1.0%
private: 1.0%
phone: 1.0%
personal: 1.0%
guide: 1.0%
GPS: 1.0%


To begin the process of boosting, we first initialize a new dictionary to store the modified distributions.

In [ ]:
boost_dist = {}

Now, we can use the `boost_gen()` function we created a few cells previous. Notice how it serves, essentially, the same programmatic role as the `torch.no_grad()` function, with the main difference being that the 20th layer of the model is "passed through" the SAE with a boosted feature.

*Note: I've added two variables you can play around with here. The first is the amount that you boost the feature. I've chosen this value (7) because I think it makes the demonstration the most clear, but if you bring it all the way up, some other interesting things happen as well.*

In [ ]:
boost_amt = 7
feature_idx = 369

for _ in range(100):
    out_ids = boost_gen(prompt1, model, tokenizer, enc_weights_fnl, enc_bias_fnl, dec_weights_fnl, dec_bias_fnl, activations_mean, activations_std, feature_idx - 1, boost_amt)

    new_id = out_ids[0][-1]
    txt = tokenizer.decode(new_id).strip()
    boost_dist[txt] = boost_dist.get(txt, 0) + 1

Finally, we can print out the distribution of responses after boosting the feature.

It seems like it was a success! As you can see, "boat" has now become the number one most frequent completion, despite only occupying a few of the non-boosted responses.

Other interesting responses that now are appearing include - "helicopter," "cruise," and "ship." It seems like our feature also inadvertently boosts the concept of maritime travel as well as air travel.

In [ ]:
sort_boost = dict(sorted(boost_dist.items(), key=lambda x: x[1], reverse=True))

print("Amount: ", boost_amt)
for word, pct in sort_boost.items():
    print(f"{word}: {pct / sum(sort_boost.values()) * 100:.1f}%")

Amount:  100
boat: 31.0%
small: 9.0%
plane: 8.0%
private: 8.0%
single: 7.0%
large: 6.0%
helicopter: 6.0%
light: 6.0%
cruise: 6.0%
fishing: 5.0%
ship: 5.0%
taxi: 5.0%
motor: 5.0%
high: 5.0%
resort: 5.0%
cargo: 4.0%
few: 4.0%
public: 4.0%
line: 4.0%
U: 4.0%
battery: 4.0%
combination: 3.0%
chain: 3.0%
diver: 3.0%
tourist: 3.0%
route: 3.0%
traffic: 3.0%
chart: 3.0%
different: 3.0%
new: 3.0%
drone: 3.0%
passenger: 3.0%
vehicle: 3.0%
separate: 2.0%
mini: 2.0%
flight: 2.0%
jet: 2.0%
non: 2.0%
charter: 2.0%
breeze: 2.0%
way: 2.0%
variety: 1.0%
road: 1.0%
personal: 1.0%
car: 1.0%
commercial: 1.0%
": 1.0%
blue: 1.0%


# (*) - Sandbox

The following section allows you to customize what feature you want to boost, how much to boost it, what prompt you want to use, and then see how the model's outputs change as a result!

Here is the same code cell I created above, which allows you to browse through all the non-dead features and pick your favorite. You can adjust the range of features you see by changing the `start_ft` and `end_ft` variables!

In [88]:
start_ft = 2360
end_ft = 2500

for i in range(start_ft - 1, end_ft):
    if wikitext_act[:, i].max().item() > 0.24:
        top_tk_tups = top_tk_only(i, wikitext_act, wikitext_tk, 10)
        print("feature", i + 1, ":", top_tk_tups)

feature 2362 : [' spent', ' during', ' spent', ' spent', ' spent', ' spent', ' during', ' During', ' spending', ' over']
feature 2363 : [' has', ' had', ' had', ' had', ' had', ' had', ' had', ' had', ' has', ' had']
feature 2366 : [' of', ' of', ' commit', ' for', ' of', ' acts', ' of', ' previous', ' of', ' were']
feature 2367 : [' located', ' located', ' located', ' located', ' position', ' stationed', ' placed', ' located', ' stationed', 'ed']
feature 2371 : [' 250', '000', ' 300', ' 400', ' 130', ' 200', ' 240', ' 800', '91', ' 150']
feature 2376 : [' tight', ' a', ' wear', ' fitting', ' long', 'earing', ' civilian', ' of', ' ,', ' wear']
feature 2380 : [' begins', ' began', ' began', ' begins', ' began', ' begin', ' began', ' began', ' began', ' begins']
feature 2387 : [' There', ' There', ' there', ' There', ' there', ' There', ' there', ' there', ' There', ' there']
feature 2397 : [' God', ' Christ', ' god', ' gods', ' Christ', ' Christ', ' God', ' Pope', ' god', ' gods']
featu

This code cell is *interactive*, and will ask you for everything you need to run your simulation. It also includes some error handling. This is highly recommended for absolute beginners, anybody who had trouble following the rest of the notebook (which is okay!), or anyone not super familiar with programming. Otherwise, feel free to skip to the next cell.

*Note: If this cell is too long, or isn't already hidden, click the arrow on the left of the "Interactive Mode" text to hide the code.*

In [79]:
#@title Interactive Mode

import textwrap
from itertools import zip_longest

textwidth = 80

layer19 = model.transformer.h[19]
layer19._forward_hooks.clear()
layer19._forward_pre_hooks.clear()
for attr in ["enc_w", "enc_b", "dec_w", "dec_b", "feat_idx", "boost_amt", "act_mean", "act_std"]:
    if hasattr(layer19, attr):
        delattr(layer19, attr)

prompt1 = "What feature would you like to boost? Please enter its index as a number with no other letters or symbols."
feature = input(textwrap.fill(prompt1, width=textwidth) + "\n\n").strip()
while not feature.isdigit() or int(feature) < 1 or int(feature) > 4096:
  prompt1_inv = "It seems like you entered a non-numeric symbol, or a number greater than the number of total features. Please provide only a number, from 1 to 4096."
  feature = input("\n" + textwrap.fill(prompt1_inv, width=textwidth) + "\n\n").strip()
tuples = top_tk_only(int(feature) - 1, wikitext_act, wikitext_tk, 10)

prompt2_q = "These are the top 10 tokens representing your feature. Do these look right? (Y/N)"
prompt2_tup = str(tuples)
answer = input("\n" + "-"*textwidth + "\n\n" + textwrap.fill(prompt2_q, width=textwidth) + "\n\n" + textwrap.fill(prompt2_tup, width=textwidth) + "\n\n").strip().lower()
while answer != "y" and answer != "n":
  answer = input("\nPlease enter either Y or N.\n\n").strip().lower()
while answer == "n":
  feature = input("\n" + textwrap.fill(prompt1, width=textwidth) + "\n\n").strip()
  while not feature.isdigit() or int(feature) < 1 or int(feature) > 4096:
    prompt2_inv = "Please provide a number between 1 and 4096."
    feature = input("\n" + textwrap.fill(prompt2_inv, width=textwidth) + "\n\n").strip()
  tuples = top_tk_only(int(feature) - 1, wikitext_act, wikitext_tk, 10)
  answer = input("\n" + textwrap.fill(prompt2_q, width=textwidth) + "\n\n" + textwrap.fill(str(tuples), width=textwidth) + "\n\n").strip().lower()
  while answer != "y" and answer != "n":
    answer = input("\nPlease enter either Y or N.\n\n").strip().lower()

prompt3_1 = "What would you like your prompt to be?"
prompt3_2 = "GPT-2 Medium is a completion model, so instead of \"What is your favorite animal,\" try \"My favorite animal is\"."
prompt = input("\n" + "-"*textwidth + "\n\n" + textwrap.fill(prompt3_1, width=textwidth) + "\n\n" + textwrap.fill(prompt3_2, width=textwidth) + "\n\n").strip()
while not prompt:
  prompt = input("\nPlease enter a character.\n\n").strip()
confirmation = input("\nAre you sure? (Y/N)\n\n").strip().lower()
while confirmation != "y" and confirmation != "n":
    confirmation = input("\nPlease enter either Y or N.\n\n").strip().lower()
while confirmation == "n":
    prompt = input("\nWhat would you like your prompt to be?\n\n").strip()
    while not prompt:
      prompt = input("\nPlease enter a non-empty prompt.\n\n").strip()
    confirmation = input("\nAre you sure? (Y/N)\n\n").strip().lower()

prompt4 = "How much would you like to boost your feature? Keep in mind that an excessive boost (~200 and above) might produce unexpected results. You can also perform a negative boost, if you'd like."
boost_amt = input("\n" + "-"*textwidth + "\n\n" + textwrap.fill(prompt4, width=textwidth) + "\n\n").strip()
while not (boost_amt.replace('.', '', 1).isdigit() or (len(boost_amt) > 1 and boost_amt[0] == '-' and boost_amt[1:].replace('.', '', 1).isdigit())):
  prompt4_inv = "It seems like you entered a non-numerical symbol. Please provide only a number."
  boost_amt = input("\n" + textwrap.fill(prompt4_inv, width=textwidth) + "\n\n").strip()
while float(boost_amt) == 0:
  prompt4_inv2 = "A boost of 0 means there will be no change! Please enter either a positive or negative number."
  boost_amt = input("\n" + textwrap.fill(prompt4_inv2) + "\n\n").strip()
  while not (boost_amt.replace('.', '', 1).isdigit() or (len(boost_amt) > 1 and boost_amt[0] == '-' and boost_amt[1:].replace('.', '', 1).isdigit())):
    prompt4_inv = "It seems like you entered a non-numerical symbol. Please provide only a number."
    boost_amt = input("\n" + textwrap.fill(prompt4_inv, width=textwidth) + "\n\n").strip()

prompt5 = "Finally, how many times would you like to run the simulation? The number you enter will apply for both before and after the feature boosting."
iter = input("\n" + "-"*textwidth + "\n\n" + textwrap.fill(prompt5, width=textwidth) + "\n\n").strip()
while not iter.isdigit() or int(iter) <= 0:
  prompt5_inv = "Please provide a positive whole number (non-zero, non-negative, and without symbols)."
  iter = input("\n" + textwrap.fill(prompt5_inv, width=textwidth) + "\n\n").strip()

print("\nPlease wait a moment while you run your simulation...")

base_dist = {}
in_tk = tokenizer(prompt, return_tensors="pt").to(model.device)
for _ in range(int(iter)):
    with torch.no_grad():
        out_ids = model.generate(**in_tk, max_new_tokens=1, pad_token_id=tokenizer.eos_token_id, do_sample=True)

    new_id = out_ids[0][-1]
    txt = tokenizer.decode(new_id).strip()
    base_dist[txt] = base_dist.get(txt, 0) + 1
sort_base = dict(sorted(base_dist.items(), key=lambda x: x[1], reverse=True))

boost_dist = {}
for _ in range(int(iter)):
    out_ids = boost_gen(prompt, model, tokenizer, enc_weights_fnl, enc_bias_fnl, dec_weights_fnl, dec_bias_fnl, activations_mean, activations_std, int(feature) - 1, float(boost_amt))

    new_id = out_ids[0][-1]
    txt = tokenizer.decode(new_id).strip()
    boost_dist[txt] = boost_dist.get(txt, 0) + 1
sort_boost = dict(sorted(boost_dist.items(), key=lambda x: x[1], reverse=True))

total_base = sum(sort_base.values())
total_boost = sum(sort_boost.values())

base_items = [(tok, cnt / total_base * 100) for tok, cnt in sort_base.items()]
boost_items = [(tok, cnt / total_boost * 100) for tok, cnt in sort_boost.items()]

max_tok = max(len(t) for t in list(sort_base.keys()) + list(sort_boost.keys()))
col_w = max_tok + 8

print("\n" + "-"*textwidth + "\n\n\"" + prompt + "...\"\n\n" + f"{'Original':<{col_w}} | {'Boosted':<{col_w}}")
print("=" * (col_w * 2 + 3))

for b, a in zip_longest(base_items, boost_items, fillvalue=("", None)):
    if b[1] is not None:
        left = f"{b[0]:<{max_tok}} {b[1]:>6.1f}%"
    else:
        left = " " * col_w
    if a[1] is not None:
        right = f"{a[0]:<{max_tok}} {a[1]:>6.1f}%"
    else:
        right = ""
    print(f"{left} | {right}")

What feature would you like to boost? Please enter its index as a number with no
other letters or symbols.

368

--------------------------------------------------------------------------------

These are the top 10 tokens representing your feature. Do these look right?
(Y/N)

[' ocean', ' sea', ' port', ' sea', ' port', ' shore', ' naval', ' naval', '
naval', ' port']

y

--------------------------------------------------------------------------------

What would you like your prompt to be?

GPT-2 Medium is a completion model, so instead of "What is your favorite
animal," try "My favorite animal is".

The best way to travel is by using a

Are you sure? (Y/N)

y

--------------------------------------------------------------------------------

How much would you like to boost your feature? Keep in mind that an excessive
boost (~200 and above) might produce unexpected results. You can also perform a
negative boost, if you'd like.

15

----------------------------------------------------

This code cell is *static*, which allows you to specify all the needed values as variables in the following cell:
1. `feature` (integer) represents the (1-) index of the feature you wish to boost.
2. `prompt` (string) represents the completion prompt you want to be sent to the model.
3. `boost_amt` (integer) represents the raw amount that you want your feature to be boosted. Be aware that an excessive boost might produce unintended (but interesting) results. You can also suppress features by specifying a negative number.
4. `iter` (integer) represents the number of times the model will be sent your prompt, for both the original (un-modified) and boosted versions.

Then, simply run the "Static Mode" code.

If you receive an error, chances are, one of your variable types is wrong. If not, make sure you haven't changed the names of the variables (since this can sometimes cause a collision with other parts of the Notebook).

*Note: If this cell is too long, click the arrow on the left of the "Static Mode" text to hide the code while it's running.*

In [89]:
feature = 369
prompt = "The best way to travel is by using a"
boost_amt = 7
iter = 100

In [90]:
#@title Static Mode

from itertools import zip_longest

layer19 = model.transformer.h[19]
layer19._forward_hooks.clear()
layer19._forward_pre_hooks.clear()
for attr in ["enc_w", "enc_b", "dec_w", "dec_b", "feat_idx", "boost_amt", "act_mean", "act_std"]:
    if hasattr(layer19, attr):
        delattr(layer19, attr)

base_dist = {}
in_tk = tokenizer(prompt, return_tensors="pt").to(model.device)
for _ in range(int(iter)):
    with torch.no_grad():
        out_ids = model.generate(**in_tk, max_new_tokens=1, pad_token_id=tokenizer.eos_token_id, do_sample=True)

    new_id = out_ids[0][-1]
    txt = tokenizer.decode(new_id).strip()
    base_dist[txt] = base_dist.get(txt, 0) + 1
sort_base = dict(sorted(base_dist.items(), key=lambda x: x[1], reverse=True))

boost_dist = {}
for _ in range(int(iter)):
    out_ids = boost_gen(prompt, model, tokenizer, enc_weights_fnl, enc_bias_fnl, dec_weights_fnl, dec_bias_fnl, activations_mean, activations_std, int(feature) - 1, float(boost_amt))

    new_id = out_ids[0][-1]
    txt = tokenizer.decode(new_id).strip()
    boost_dist[txt] = boost_dist.get(txt, 0) + 1
sort_boost = dict(sorted(boost_dist.items(), key=lambda x: x[1], reverse=True))

total_base = sum(sort_base.values())
total_boost = sum(sort_boost.values())

base_items = [(tok, cnt / total_base * 100) for tok, cnt in sort_base.items()]
boost_items = [(tok, cnt / total_boost * 100) for tok, cnt in sort_boost.items()]

max_tok = max(len(t) for t in list(sort_base.keys()) + list(sort_boost.keys()))
col_w = max_tok + 8

print("\"" + prompt + "...\"\n\n" + f"{'Original':<{col_w}} | {'Boosted':<{col_w}}")
print("=" * (col_w * 2 + 3))

for b, a in zip_longest(base_items, boost_items, fillvalue=("", None)):
    if b[1] is not None:
        left = f"{b[0]:<{max_tok}} {b[1]:>6.1f}%"
    else:
        left = " " * col_w
    if a[1] is not None:
        right = f"{a[0]:<{max_tok}} {a[1]:>6.1f}%"
    else:
        right = ""
    print(f"{left} | {right}")

"The best way to travel is by using a..."

Original            | Boosted            
car           17.0% | boat          16.0%
bike          12.0% | motor          7.0%
bicycle       11.0% | helicopter     5.0%
bus            5.0% | sea            4.0%
motorcycle     5.0% | small          4.0%
private        4.0% | vehicle        4.0%
vehicle        4.0% | single         4.0%
train          3.0% | ship           3.0%
public         3.0% | large          3.0%
GPS            3.0% | cruise         3.0%
cab            2.0% | chain          3.0%
guide          2.0% | plane          3.0%
map            2.0% | taxi           3.0%
combination    2.0% | light          2.0%
regular        2.0% | non            2.0%
taxi           2.0% | jet            2.0%
travel         2.0% | cargo          2.0%
boat           2.0% | fleet          2.0%
local          2.0% | new            2.0%
rail           2.0% | high           2.0%
ride           2.0% | chart          2.0%
carrier        1.0% | charter    

**Have fun!**

I hope this was informative. If you have any suggestions for things I could add to this Notebook, things I could have explained better, or anything else, please reach out via email (danielliu.datascience@gmail.com)!